# 08 · Lookup & Query App
Name search + NL questions. Gemini (offline: deterministic parser) translates a question into a STRUCTURED query plan; deterministic code executes it over the tables (the LLM plans, the tables answer). Dossiers render with clickable evidence that highlights the exact span in the raw note. A static HTML snapshot is exported.

In [ ]:
# --- bootstrap: make the src package importable from any working dir ---
import sys
from pathlib import Path
p = Path.cwd().resolve()
while not (p / 'config' / '00_config.py').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
print('project root:', p)


In [ ]:
import os, json
from src.repository import Repository
from src import app
repo = Repository()
index = app.EntityIndex(repo)
print('dossiers indexed:', len(index.dossiers))


In [ ]:
# run the preloaded example questions: show the generated plan + table-executed answer
for q in app.EXAMPLE_QUESTIONS:
    out = app.answer_question(repo, index, q)
    names = [index.dossiers[e]['canonical_name'] for e in out['result']['entity_ids'][:3]]
    print('Q:', q)
    print('   plan   :', json.dumps(out['plan']))
    print('   answer : n =', out['result']['n'], '| top:', names)
    print()


In [ ]:
# export a self-contained clickable-evidence dossier snapshot
eid = max(index.dossiers, key=lambda e: index.dossiers[e]['n_mentions'])
path = app.export_dossier_html(repo, eid)
print('exported dossier snapshot ->', path)
print('open this file in a browser: click any evidence item to jump to the highlighted span.')


In [ ]:
# launch the Gradio app (skipped automatically under headless orchestration)
import os
if os.environ.get('LAUNCH_APP', '1') != '0':
    demo = app.build_app(repo)
    demo.launch(share=False)
else:
    print('LAUNCH_APP=0 -> skipping interactive launch (headless run).')
